# Anàlisi general taules de dades csv PADRIS

In [2]:
import pandas as pd
import numpy as np
import random as rd

## 1. Taula 5 Derivació dels pacients

In [4]:
path = r'C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets\689_F_ap_derivacio_sm_Final.csv'
df_derivacio = pd.read_csv(path)

In [5]:
df_derivacio.head()

,id,idCas,Identificador de registre CMBD,Unitat Proveïdora,UP_descripcio,Data inici assistència,Tipus de professional,Tipus_professional_descrip,Unitat Proveïdora Destí,Tipus UP Destí
0,1,83892072,263736108,337,EAP Baix Berguedà,3/27/2015,1,1 Metge/ssa de família,CSM Adults Berguedà,ASSISTÈNCIA SALUT MENTAL - AMBULATÒRIA
1,2,83892237,306747946,1933,EAP Barcelona 7B - Sardenya,10/21/2015,1,1 Metge/ssa de família,CSM Adults Guinardó,ASSISTÈNCIA SALUT MENTAL - AMBULATÒRIA
2,3,83897885,250764528,294,EAP Badalona 3 - Progrés-Raval,1/8/2015,1,1 Metge/ssa de família,CSM Adults Badalona 1 Est,ASSISTÈNCIA SALUT MENTAL - AMBULATÒRIA
3,4,83900256,278190986,157,EAP Garraf Rural,7/9/2015,1,1 Metge/ssa de família,CSM Garraf,ASSISTÈNCIA SALUT MENTAL - AMBULATÒRIA
4,5,83902712,257236651,1798,EAP Palamós,2/19/2015,2,2 Pediatre/a,CSM Adults Baix Empordà,ASSISTÈNCIA SALUT MENTAL - AMBULATÒRIA


In [6]:
df_derivacio.Tipus_professional_descrip.unique()

array(['1 Metge/ssa de família', '2 Pediatre/a', '4 Infermer/a',
       '3 Odontòleg/oga', '5 Treballador/a social'], dtype=object)

In [7]:
df_derivacio["Tipus de professional"].unique()

array([1, 2, 4, 3, 5])

In [8]:
df_derivacio.id.count()

np.int64(7195)

## 2. Tabac

VAL:
- 0= No fumador 
- 1= Fumador 
- 2 = exfumador

In [9]:
df_tabac=pd.read_csv(r"C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets\8_tabac_cohort_total_Final.csv")
df_tabac.head()

,id,idCas,DAT,DBAIXA,VAL
0,1,73405423,2012-11-12T22:00:00.000Z,2017-12-30T22:00:00.000Z,0
1,2,73405425,2016-06-08T20:00:00.000Z,2017-12-30T22:00:00.000Z,0
2,3,73405429,2016-10-23T20:00:00.000Z,2017-12-30T22:00:00.000Z,0
3,4,73405436,2017-03-05T22:00:00.000Z,2017-12-30T22:00:00.000Z,0
4,5,78465847,2017-11-07T22:00:00.000Z,2017-12-30T22:00:00.000Z,0


In [10]:
df_tabac.VAL.unique()

array([0, 2, 1])

### Columnes triades: 'VAL'

## Fàrmacs Dispensats

In [1]:
import pandas as pd
import numpy as np
import re

# Codis ATC d'antipsicòtics
ANTIPSICIOTICS_ATC = (
    'N05AD01', 'N05AD1',   # Haloperidol
    'N05AH02',              # Clozapina
    'N05AH03',              # Olanzapina
    'N05AH04',              # Quetiapina
    'N05AH06',              # Clotiapina
    'N05AL01',              # Sulpirida
    'N05AL05',              # Amisulpirida
    'N05AX08',              # Risperidona
    'N05AX12',              # Aripiprazol
    'N05AX13',              # Paliperidona
    'N05AE05',              # Lurasidona
)

def es_antipsicotic(serie_atc: pd.Series) -> pd.Series:
    """Retorna màscara booleana: True si el codi ATC és un antipsicòtic."""
    return serie_atc.str.startswith(ANTIPSICIOTICS_ATC, na=False)

#  Funció per llegir en chunks i filtrar al vol 
def llegir_filtrant(path: str, cols: list[str], chunk_size: int = 50_000) -> pd.DataFrame:
    """
    Llegeix un CSV gran en trossos (chunks) i només guarda
    les files d'antipsicòtics. Molt més eficient en memòria.
    """
    chunks_filtrats = []
    
    for chunk in pd.read_csv(
        path,
        sep=',',
        encoding='latin1',
        usecols=cols,           # <-- llegeix NOMÉS les columnes que necessites
        chunksize=chunk_size,
        low_memory=False
    ):
        mask = es_antipsicotic(chunk['ID_HIS_Subgrup_7_ATC'])
        chunks_filtrats.append(chunk[mask])
    
    return pd.concat(chunks_filtrats, ignore_index=True)

### OF (Oficina de farmàcia)
23_farmacs_dispensants_2010_2019_Final

In [18]:
import pandas as pd

path_farmacs_dispensats = r"C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets\23_farmacs_dispensants_2010_2019_Final.csv"

cols = pd.read_csv(path_farmacs_dispensats, sep=',', encoding='latin1', nrows=0).columns.tolist()
print(cols)

['id', 'idCas', 'Any_Mes', 'ID_HIS_Subgrup_7_ATC', 'DES_HIS_Subgrup_7_ATC', 'ID_HIS_Producte', 'DES_HIS_Producte', 'HIS_Quantitat_PA_Espec', 'ID_HIS_Unitat_Mesura_PA_Espec', 'DES_HIS_Unitat_Mesura_PA_Espec', 'HIS_Unitats_Productes', 'ID_HIS_Forma_Farmaceutica', 'DES_HIS_Forma_Farmaceutica', 'ID_HIS_Via_Administracio_Producte', 'DES_HIS_Via_Administracio_Producte', 'Nombre_Receptes_Dispensades', 'Nombre_DDD_Receptes_Dispensades', 'Import_Integre_Dispensades', 'Import_Liquid_Dispensat', 'Nombre_Envasos_Dispensats']


In [7]:
ANTIPSICIOTICS_ATC = (
    'N05AD01', 'N05AD1',   # Haloperidol
    'N05AH02',              # Clozapina
    'N05AH03',              # Olanzapina
    'N05AH04',              # Quetiapina
    'N05AH06',              # Clotiapina
    'N05AL01',              # Sulpirida
    'N05AL05',              # Amisulpirida
    'N05AX08',              # Risperidona
    'N05AX12',              # Aripiprazol
    'N05AX13',              # Paliperidona
    'N05AE05',              # Lurasidona
)

BASE = r"C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets"

def llegir_filtrant(path):
    chunks = []
    for chunk in pd.read_csv(path, sep=',', encoding='latin1', chunksize=50_000, low_memory=False):
        mask = chunk['ID_HIS_Subgrup_7_ATC'].str.startswith(ANTIPSICIOTICS_ATC, na=False)
        chunks.append(chunk[mask])
    return pd.concat(chunks, ignore_index=True)

df_dispensats = llegir_filtrant(f"{BASE}\\23_farmacs_dispensants_2010_2019_Final.csv")

print(df_dispensats.head())

   id     idCas  Any_Mes ID_HIS_Subgrup_7_ATC DES_HIS_Subgrup_7_ATC  \
0   3  68121038   201912              N05AX12           Aripiprazol   
1   7  68121038   201912              N05AX12           Aripiprazol   
2  12  69210559   201912              N05AX13          Paliperidona   
3  13  69210559   201911              N05AH04            Quetiapina   
4  14  69210559   201911              N05AX13          Paliperidona   

   ID_HIS_Producte                                   DES_HIS_Producte  \
0           728196                        ABILIFY 10MG 28 COMPRIMIDOS   
1           728196                        ABILIFY 10MG 28 COMPRIMIDOS   
2           723799  INVEGA 3MG 28 COMPRIMIDOS DE LIBERACION PROLON...   
3           661762  QUETIAPINA STADA 100MG 60 COM RE P BLIS PVC/AI...   
4           723799  INVEGA 3MG 28 COMPRIMIDOS DE LIBERACION PROLON...   

  HIS_Quantitat_PA_Espec ID_HIS_Unitat_Mesura_PA_Espec  \
0                     10                             4   
1                 

In [12]:
df_farmacs_dispensats=pd.read_csv(r"C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets\23_farmacs_dispensants_2010_2019_Final.csv", sep=',', encoding='latin1')

C:\Users\LRAJAG\AppData\Local\Temp\ipykernel_20412\4087764429.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_farmacs_dispensats=pd.read_csv(r"C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets\23_farmacs_dispensants_2010_2019_Final.csv", sep=',', encoding='latin1')


In [14]:
total_files = sum(len(chunk) for chunk in pd.read_csv(path_farmacs_dispensats, sep=',', encoding='latin1', chunksize=50_000, low_memory=False, usecols=['idCas']))

print(f"Farmacs Dispensats Total:    {total_files:,}")
print(f"Antipsicotics Dispensats Total: {len(df_dispensats):,}")

Farmacs Dispensats Total:    67,086,050
Antipsicotics Dispensats Total: 8,140,469


In [15]:
df_dispensats.to_csv(r'C:\Users\LRAJAG\Desktop\Data_sintetica\src\ADHERENCIA\antipsicotics_dispensats.csv', index=False)

### MHDA (Hospital)
Seleccionar mateixes columnes que farmacs OF però afegim pas de treure pacients amb idCas dins de cohort_control.
Pel futur merge amb farmacs_dispensats afegim columna d'envasos que és majoritàriament 1.

In [17]:
import pandas as pd

path_mhda = r"C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets\25_taula_farmacs_mhda_2010_2019_Final.csv"

cols = pd.read_csv(path_farmacs_dispensats, sep=',', encoding='latin1', nrows=0).columns.tolist()
print(cols)

['id', 'idCas', 'Any_Mes', '(HIS) Subgrup 7 (ATC)', '(HIS) Subgrup 7 (ATC) Desc', '(HIS) Forma FarmacÃ¨utica', '(HIS) Forma FarmacÃ¨utica Desc', '(HIS) Producte', '(HIS) Producte Desc', '(HIS) Via de distribuciÃ³', 'Concepte FacturaciÃ³', 'Concepte FacturaciÃ³ Desc', 'Import activitat MHDA']


In [20]:
total_files_mhda = 0
chunks_filtrats_mhda = []

for chunk in pd.read_csv(path_mhda, sep=',', encoding='latin1', chunksize=50_000, low_memory=False):
    total_files_mhda += len(chunk)
    mask = chunk['(HIS) Subgrup 7 (ATC)'].str.startswith(ANTIPSICIOTICS_ATC, na=False)
    chunks_filtrats_mhda.append(chunk[mask])

df_mhda = pd.concat(chunks_filtrats_mhda, ignore_index=True)

print(f"MHDA Total:          {total_files_mhda:,}")
print(f"MHDA Antipsicotics:  {len(df_mhda):,}")

MHDA Total:          837,826
MHDA Antipsicotics:  2,580


In [21]:
df_mhda.to_csv(r'C:\Users\LRAJAG\Desktop\Data_sintetica\src\ADHERENCIA\antipsicotics_mhda.csv', index=False)